# FaceForge — Phase 0: Environment and Data Pipeline

**Goal:** Install dependencies, download CelebA, build the dataset class, visualize a batch, and compute attribute statistics.

Run all cells top-to-bottom. Each cell is self-contained and safe to re-run after session resume.

## Cell 0 — Mount Google Drive
All data and checkpoints are stored on Drive so they survive session restarts.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os

# # All persistent data lives here. Change if your Drive layout differs.
# DRIVE_ROOT   = '/content/drive/MyDrive/FaceForge'
# DATA_ROOT    = os.path.join(DRIVE_ROOT, 'data')       # CelebA lives here
# CKPT_ROOT    = os.path.join(DRIVE_ROOT, 'checkpoints')
# REPO_ROOT    = '/content/FaceForge'                   # cloned repo

# os.makedirs(DATA_ROOT, exist_ok=True)
# os.makedirs(CKPT_ROOT, exist_ok=True)

# print(f'Drive mounted. Data root : {DATA_ROOT}')
# print(f'Checkpoint root          : {CKPT_ROOT}')

## Cell 1 — Install Dependencies

In [ ]:
# torch and torchvision are pre-installed on Colab; we only add the extras.
# !pip install -q ftfy clip-anytorch scikit-image matplotlib
import os, subprocess
import sys
import torch, torchvision
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

PyTorch  : 2.11.0+cpu
CUDA     : None
GPU      : None


## Cell 2 — Clone / Pull Repo

In [ ]:
# REPO_URL = 'https://github.com/YOUR_USERNAME/FaceForge.git'  # TODO: update URL

# if os.path.isdir(REPO_ROOT):
#     print('Repo exists — pulling latest...')
#     subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=True)
# else:
#     print('Cloning repo...')
#     subprocess.run(['git', 'clone', REPO_URL, REPO_ROOT], check=True)

# if REPO_ROOT not in sys.path:
#     sys.path.insert(0, REPO_ROOT)

# print('Repo ready at', REPO_ROOT)

## Cell 3 — Download CelebA

**Primary path:** `torchvision.datasets.CelebA` with `download=True`.  
**Fallback:** If Google Drive throttles the download, follow the manual instructions printed below.

In [ ]:
from data_pipeline import celeba_is_downloaded
import torchvision

if celeba_is_downloaded(DATA_ROOT):
    print('CelebA already downloaded — skipping.')
else:
    print('Attempting automatic download via torchvision...')
    print('This can fail with a Google Drive quota error. If it does, see fallback below.\n')
    torchvision.datasets.CelebA(
        root=DATA_ROOT,
        split='train',
        target_type='attr',
        download=True
    )

## Cell 4 — Build DataLoader and Verify Shapes

In [ ]:
from data_pipeline import make_dataloader, SELECTED_ATTRS, NUM_ATTRS
import torch

train_loader = make_dataloader(
    root=DATA_ROOT,
    split='train',
    download=False,
    batch_size=64,
    num_workers=2
)

images, attrs = next(iter(train_loader))

print(f'Image batch : shape={images.shape}  dtype={images.dtype}  '
      f'min={images.min():.2f}  max={images.max():.2f}')
print(f'Attr  batch : shape={attrs.shape}   dtype={attrs.dtype}')
print(f'Selected attributes ({NUM_ATTRS}): {SELECTED_ATTRS}')

# Hard assertions — these must pass before proceeding
assert images.shape == (64, 3, 64, 64), f'Bad image shape: {images.shape}'
assert attrs.shape  == (64, 18),        f'Bad attr shape: {attrs.shape}'
assert attrs.dtype  == torch.float32,   f'Attrs should be float32, got {attrs.dtype}'
assert images.min() >= -1.01 and images.max() <= 1.01, 'Images not in [-1, 1]'

print('\nAll shape/dtype assertions passed.')

## Cell 5 — Visualize a 4×4 Image Grid

In [ ]:
from data_pipeline import visualize_batch

grid_save_path = os.path.join(DRIVE_ROOT, 'phase0_batch_grid.png')

# Display inline in the notebook
visualize_batch(images, attrs, n_rows=4, n_cols=4)

# Also save to Drive for reference
visualize_batch(images, attrs, n_rows=4, n_cols=4, save_path=grid_save_path)

## Cell 6 — Attribute Statistics

Computes the mean (prevalence) and std of each attribute across all 162,770 training images.
Useful for understanding class imbalance before training.

In [ ]:
from data_pipeline import compute_attr_statistics, print_attr_statistics
import torch

# Use a stats-only loader: no shuffle, larger batch for speed
stats_loader = make_dataloader(
    root=DATA_ROOT, split='train', download=False,
    batch_size=256, num_workers=2
)

stats = compute_attr_statistics(stats_loader)
print_attr_statistics(stats)

# Save stats to Drive
stats_path = os.path.join(DRIVE_ROOT, 'phase0_attr_stats.pt')
torch.save(stats, stats_path)
print(f'\nStats saved to {stats_path}')

## Cell 7 — Phase 0 Validation Summary

Run this cell last. If all checks pass, Phase 0 is complete. Notify the human to verify the checklist.

In [ ]:
import torch
from data_pipeline import make_dataloader, NUM_ATTRS

print('=== Phase 0 Validation ===')
print()

# Check 1: DataLoader iterates 3 batches without error
loader = make_dataloader(root=DATA_ROOT, split='train',
                         download=False, batch_size=64, num_workers=2)
for i, (imgs, atts) in enumerate(loader):
    if i == 2:
        break
print('[PASS] DataLoader iterated 3 batches without error')

# Check 2: Image shape and range
assert imgs.shape[1:] == (3, 64, 64)
assert imgs.min() >= -1.01 and imgs.max() <= 1.01
print(f'[PASS] Image shape {imgs.shape}, range [{imgs.min():.2f}, {imgs.max():.2f}]')

# Check 3: Attribute shape and dtype
assert atts.shape[1] == NUM_ATTRS and atts.dtype == torch.float32
print(f'[PASS] Attribute shape {atts.shape}, dtype {atts.dtype}')

# Check 4: Stats file exists
stats_path = os.path.join(DRIVE_ROOT, 'phase0_attr_stats.pt')
assert os.path.exists(stats_path), f'Stats not found at {stats_path}'
print(f'[PASS] Attribute stats saved at {stats_path}')

# Check 5: Grid image exists
grid_path = os.path.join(DRIVE_ROOT, 'phase0_batch_grid.png')
assert os.path.exists(grid_path), f'Grid image not found at {grid_path}'
print(f'[PASS] Batch grid saved at {grid_path}')

print()
print('All automated checks passed.')
print()
print('Human validation checklist:')
print('  [ ] Images display correctly (faces visible, no corruption)')
print('  [ ] Attribute vector shape is [18], dtype is float32')
print('  [ ] DataLoader iterates without memory errors')
print('  [ ] Drive mounting and checkpoint path confirmed')
print()
print('Awaiting human approval before proceeding to Phase 1.')